# Exercise - Treasury Arbitrage

#### Notation Commands

$$\newcommand{\Black}{\mathcal{B}}
\newcommand{\Blackcall}{\Black_{\mathrm{call}}}
\newcommand{\Blackput}{\Black_{\mathrm{put}}}
\newcommand{\EcondS}{\hat{S}_{\mathrm{conditional}}}
\newcommand{\Efwd}{\mathbb{E}^{T}}
\newcommand{\Ern}{\mathbb{E}^{\mathbb{Q}}}
\newcommand{\Tfwd}{T_{\mathrm{fwd}}}
\newcommand{\Tunder}{T_{\mathrm{bond}}}
\newcommand{\accint}{A}
\newcommand{\carry}{\widetilde{\cpn}}
\newcommand{\cashflow}{C}
\newcommand{\convert}{\phi}
\newcommand{\cpn}{c}
\newcommand{\ctd}{\mathrm{CTD}}
\newcommand{\disc}{Z}
\newcommand{\done}{d_{1}}
\newcommand{\dt}{\Delta t}
\newcommand{\dtwo}{d_{2}}
\newcommand{\flatvol}{\sigma_{\mathrm{flat}}}
\newcommand{\flatvolT}{\sigma_{\mathrm{flat},T}}
\newcommand{\float}{\mathrm{flt}}
\newcommand{\freq}{m}
\newcommand{\futprice}{\mathcal{F}(t,T)}
\newcommand{\futpriceDT}{\mathcal{F}(t+h,T)}
\newcommand{\futpriceT}{\mathcal{F}(T,T)}
\newcommand{\futrate}{\mathscr{f}}
\newcommand{\fwdprice}{F(t,T)}
\newcommand{\fwdpriceDT}{F(t+h,T)}
\newcommand{\fwdpriceT}{F(T,T)}
\newcommand{\fwdrate}{f}
\newcommand{\fwdvol}{\sigma_{\mathrm{fwd}}}
\newcommand{\fwdvolTi}{\sigma_{\mathrm{fwd},T_i}}
\newcommand{\grossbasis}{B}
\newcommand{\hedge}{\Delta}
\newcommand{\ivol}{\sigma_{\mathrm{imp}}}
\newcommand{\logprice}{p}
\newcommand{\logyield}{y}
\newcommand{\mat}{(n)}
\newcommand{\nargcond}{d_{1}}
\newcommand{\nargexer}{d_{2}}
\newcommand{\netbasis}{\tilde{\grossbasis}}
\newcommand{\normcdf}{\mathcal{N}}
\newcommand{\notional}{K}
\newcommand{\pfwd}{P_{\mathrm{fwd}}}
\newcommand{\pnl}{\Pi}
\newcommand{\price}{P}
\newcommand{\probexer}{\hat{\mathcal{P}}_{\mathrm{exercise}}}
\newcommand{\pvstrike}{K^*}
\newcommand{\refrate}{r^{\mathrm{ref}}}
\newcommand{\rrepo}{r^{\mathrm{repo}}}
\newcommand{\spotrate}{r}
\newcommand{\spread}{s}
\newcommand{\strike}{K}
\newcommand{\swap}{\mathrm{sw}}
\newcommand{\swaprate}{\cpn_{\swap}}
\newcommand{\tbond}{\mathrm{fix}}
\newcommand{\ttm}{\tau}
\newcommand{\value}{V}
\newcommand{\vega}{\nu}
\newcommand{\years}{\tau}
\newcommand{\yearsACT}{\tau_{\mathrm{act/360}}}
\newcommand{\yield}{Y}$$

# 1. Treasury Arbitrage

Consider the following market data as of `Dec 29, 2023`.

The table below shows two Treasury securities, a T-note and a T-bond. They mature on the same date.

In [2]:
import pandas as pd
from pandas import IndexSlice

summary = pd.DataFrame(index=[],columns = [207391,204095],dtype=float)
summary.loc['issue date'] = ['2019-08-15','1999-08-15']
summary.loc['maturity date'] = ['2029-08-15','2029-08-15']
summary.loc['coupon rate'] = [.01625, .06125]
summary.loc['clean price'] = [89.03125,111.0391]
summary.loc['accrued interest'] = [.6005, 2.2636]
summary.loc['ytm'] = [.037677, .038784]

summary.style.format("{:.2%}", subset=IndexSlice[["ytm","coupon rate"], :]).format("{:.2f}", subset=IndexSlice[["accrued interest","clean price"], :])

,207391,204095
issue date,2019-08-15,1999-08-15
maturity date,2029-08-15,2029-08-15
coupon rate,1.62%,6.12%
clean price,89.03,111.04
accrued interest,0.60,2.26
ytm,3.77%,3.88%


As usual, the quotes are per $100 face value.

### 1.1.

Which bond would you go long, and which bond would you short?

Explain your reasoning.

Long 204095 and short 207391
- Reasoning: Although both have the same maturity date, 204095 has a higher YTM (3.88%), meaning that it is cheaper relative to its cashflows. In other words, we would earn more yield for buying 204095 than 207391. By shorting 207391 we are selling a more expensive bond and hope that its price will fall and the gap between the two bonds will narrow over time.

### 1.2.

Explain how you would finance the trade? Be specific in explaining how you would raise the cash for the long position and how you would achieve the short position.

- Long Position: Finance the purchase of 204095 thorugh a repo agreement where the bond is collateral and we would receive cash equal to the bond's market value less a haircut. Then, we could use the borrowed cash to fund the bond purchase and pay the repo rate on the borrowed cash.
- Short Position: Borrow 207391 through a reverse repo transaction where we'd sell the borrowed bond at its current market price. The cash from the sale will go towards either collateral and/or financing the long position. Then, while the short is open, we pay the coupon on 207391 to the bond lender. 

### 1.3. 

What are the risks of this trade? Is it an arbitrage in the short-term or in the long-term? Explain.

Arbitrage in the long-term
- Explanation: If held to maturity, both bonds pay the same principal at the same date, and the relative mispricing must converge. In the short-term, on the other hand, prices can move against you due to risks including financing, liquidity, and convergence risks. On the other hand, in the long run, relative prices and yields are forced to converge by maturity and the short-term risks identified earlier must dissappear. 

### 1.4.

Suppose that on `2024-02-15`, immediately after the coupon is paid out, we observe the prices are `87` and `113`, respectively.
* Approximate time-to-maturity as 5.5 years.
* Note that the coupon was just paid, so there is no accrued interest.

Calculate the new YTMs for the two bonds.

In [3]:
import numpy as np
from scipy.optimize import brentq

FACE = 100
FREQ = 2
T = 5.5
N = int(round(T * FREQ))  # 11 periods

def ytm_from_price(price, coupon_rate):
    c = FACE * coupon_rate / FREQ  # semiannual coupon dollars

    def pv_diff(y):
        r = y / FREQ  # per-period yield
        dfs = 1 / (1 + r) ** np.arange(1, N + 1)
        pv_coupons = c * dfs.sum()
        pv_face = FACE * dfs[-1]
        return (pv_coupons + pv_face - price)

    # Solve for y in a reasonable range (0% to 20%)
    return brentq(pv_diff, 1e-8, 0.20)

ytm_207391 = ytm_from_price(price=87, coupon_rate=0.0162)  # 1.62% coupon
ytm_204095 = ytm_from_price(price=113, coupon_rate=0.0612) # 6.12% coupon

new_ytms = pd.DataFrame(index=['New YTM'], columns=[207391,204095], dtype=float)
new_ytms.loc['New YTM', 207391] = ytm_207391
new_ytms.loc['New YTM', 204095] = ytm_204095
display(new_ytms.style.format("{:.3%}"))


,207391,204095
New YTM,4.299%,3.501%


### 1.5.

Suppose that on `2024-08-15` we observe the YTMs are now 4.65% and 4.70%, respectively. 

Note that this observation is immediately **after** the bonds pay out coupons.

* Calculate the prices of both bonds.
* Compare this to the **dirty** prices inferred from the table above.


In [10]:
FACE = 100
FREQ = 2
N = 10  # 5 years * 2

def bond_price(ytm, coupon_rate):
    c = FACE * coupon_rate / FREQ
    r = ytm / FREQ
    dfs = 1 / (1 + r) ** np.arange(1, N + 1)
    price = c * dfs.sum() + FACE * dfs[-1]
    return price

price_207391 = bond_price(ytm=0.0465, coupon_rate=0.0162)
price_204095 = bond_price(ytm=0.0470, coupon_rate=0.0612)


dirty_prices = {
    "207391": 89.03 + 0.60,   # 89.63
    "204095": 111.04 + 2.26   # 113.30
}

df_prices = pd.DataFrame(
    {
        "Dirty Price (Dec 29, 2023)": [dirty_prices["207391"], dirty_prices["204095"]],
        "Price (Aug 15, 2024)": [price_207391, price_204095],
    },
    index=["207391", "204095"]
)

df_prices["Price Change"] = (
    df_prices["Price (Aug 15, 2024)"] - df_prices["Dirty Price (Dec 29, 2023)"]
)

display(df_prices.style.format("{:.3f}"))


,"Dirty Price (Dec 29, 2023)","Price (Aug 15, 2024)",Price Change
207391,89.630,86.620,-3.010
204095,113.300,106.262,-7.038


Both bond prices fall as the yields increased between 12/2023 and 08/2024, with the high-coupon bond falling more. Even though high-coupon bonds have slightly lower duration, their higher price level means larger dollar moves for similar yield changes.

### 1.6.

Suppose that you have a position...
* long 1 unit of T-bond (ID=`204095`).
* short 1 unit of T-note (ID=`207391`).

(Note that we're not balancing the dollars or risk. Just assume long-short one unit of each.)


Calculate the value of this long-short position using the 
* prices in the given table
* prices from `1.5`

How did the value of the position change? 

In [11]:
#Dirty prices from the original table
dirty_207391 = 89.03 + 0.60
dirty_204095 = 111.04 + 2.26

#Value of position
V_dec2023 = dirty_204095 - dirty_207391
V_aug2024 = price_204095 - price_207391
dV = V_aug2024 - V_dec2023

df_position = pd.DataFrame(
    {
        "204095 Price (Long)": [dirty_204095, price_204095],
        "207391 Price (Short)": [dirty_207391, price_207391],
        "Long-Short Value": [V_dec2023, V_aug2024],
    },
    index=["12/29/2023", "8/15/2024"]
)

df_position["Change in Position Value"] = [
    None,
    dV
]

df_position.round(4)


,204095 Price (Long),207391 Price (Short),Long-Short Value,Change in Position Value
12/29/2023,113.3000,89.6300,23.6700,NaN
8/15/2024,106.2624,86.6199,19.6425,-4.0275


The value of the long-short position declines by about 4.03 points, which is an unfavorable relative price movement. This divergence demonstrates how the Treasury arbitrage is not risk-free in the short run.

***